# PlugIR S3 selector comparison · Hits@10

All four methods use the same 200 dialogues and 50,000-image index. Each dialogue stops at its first user-facing top-10 hit or after 10 questions. Round 0 is the search before any questions.

- **Vanilla** uses PlugIR’s k-means and entropy-based representatives.
- **Fused Top 10** uses the 10 captions with highest reciprocal rank fusion scores from all distinct queries so far.
- **Current Top 10** uses the 10 highest-ranked captions from the latest query.
- **History MMR** uses the fused ranking for relevance and penalizes redundancy using cosine similarity between BLIP image embeddings.

All four use the latest query’s candidate pool and user-facing ranking. The notebook reuses the completed vanilla and Fused Top 10 checkpoints. The two new selectors run concurrently on Apple Silicon. The progress and plot cells can read saved results on a CPU.

In [ ]:
from pathlib import Path
import json
import sys
import subprocess

# Start Jupyter in the directory containing this notebook, or set PACKAGE explicitly.
PACKAGE = Path.cwd().resolve()
RESULTS = PACKAGE / "results"
REPORT = PACKAGE / "source" / "report_hits.py"

# Change these paths if your checkout or MLX environment lives elsewhere.
PROJECT = next(
    (p for p in (PACKAGE, *PACKAGE.parents) if (p / "ChatIR-Cornell2026").is_dir()),
    PACKAGE.parent,
)
REPO = next(
    (p for p in (PACKAGE, *PACKAGE.parents) if (p / ".git").exists()),
    PROJECT / "ChatIR-Cornell2026",
)
BLIP_PYTHON = REPO / ".venv" / "bin" / "python"
MLX_ENV = PROJECT / "tmp" / "agentir-mlx-vlm-env"
QWEN_REVISION = "8b2b98c00a6b4d291155e4890773ca8f769aee53"
SNAPSHOT = (REPO / "cache/huggingface/hub/models--mlx-community--Qwen3.5-9B-4bit"
            / "snapshots" / QWEN_REVISION)
ARMS = ["s3_current_top10", "s3_history_mmr"]
REFERENCE_ARMS = ["vanilla", "s3_fused_top10"]
REPORT_ARMS = [*REFERENCE_ARMS, *ARMS]
SAMPLE = json.loads((PACKAGE / "sample.json").read_text())
DIALOGUE_COUNT = len(SAMPLE["tasks"])
print(f"{DIALOGUE_COUNT} dialogues per method, seed {SAMPLE['seed']}")

## Check saved progress

A file is written for every completed dialogue. The matched count includes IDs with a saved result for all four methods.

In [ ]:
def saved_ids(arm):
    folder = RESULTS / arm
    if arm in ARMS:
        folder /= arm
    return {int(path.stem) for path in (folder / "dialogues").glob("*.json")}

for arm in REPORT_ARMS:
    print(f"{arm}: {len(saved_ids(arm))}/{DIALOGUE_COUNT}")
print(f"Matched: {len(set.intersection(*(saved_ids(arm) for arm in REPORT_ARMS)))}/{DIALOGUE_COUNT}")

## Run or resume on Apple Silicon

Run this cell to start both new variants in parallel. If they are already running, use the progress cell. Set `RESULTS` to a new directory for a separate replication. The run cell copies the included reference checkpoints to a fresh results directory.

In [ ]:
from shutil import copytree

for dependency in (BLIP_PYTHON, MLX_ENV / "bin/python", SNAPSHOT / "config.json"):
    if not dependency.exists():
        raise FileNotFoundError(f"{dependency} is missing. See README.md for setup.")

reference = PACKAGE / "results"
if RESULTS.resolve() != reference.resolve():
    for arm in REFERENCE_ARMS:
        copytree(reference / arm, RESULTS / arm, dirs_exist_ok=True)

subprocess.run(
    ["caffeinate", "-i", str(BLIP_PYTHON), "-u", str(PACKAGE / "source/run_selectors.py"),
     "--repo", str(REPO), "--mlx-env", str(MLX_ENV), "--snapshot", str(SNAPSHOT),
     "--results", str(RESULTS)],
    check=True,
)

## Plot the matched comparison

The default requires all 200 dialogues across the four methods. Set `ALLOW_PARTIAL = True` to inspect an interim curve labeled with its smaller sample size.

In [ ]:
from IPython.display import Image, display

ALLOW_PARTIAL = False
command = [sys.executable, str(REPORT), str(RESULTS)]
if ALLOW_PARTIAL:
    command.append("--allow-partial")
figures = subprocess.check_output(command, text=True).strip()
display(Image(filename=str(Path(figures) / "hits_at_10.png")))